<a href="https://colab.research.google.com/github/Shaunak-Mukherjee/ECE-59500---Reinforcement-Learning--Theory-and-Algorithms/blob/main/ECE_59500_Reinforcement_Learning_HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Answer 1.1:**

In [ ]:
import numpy as np
from scipy.linalg import solve
from IPython.display import display, Markdown

# Define MDP
states = [1, 2, 3]
actions = ['g', 'h']
gamma = 0.95
initial_state = 1

# Transition probabilities P(s'|s,a) -> Format: P[s][a][s']
P = {
  1: {'g': {1: 0.1, 2: 0.8, 3: 0.1}, 'h': {1: 0.8, 2: 0.1, 3: 0.1}},
  2: {'g': {1: 0.1, 2: 0.1, 3: 0.8}, 'h': {1: 0.1, 2: 0.8, 3: 0.1}},
  3: {'g': {1: 0.8, 2: 0.1, 3: 0.1}, 'h': {1: 0.1, 2: 0.1, 3: 0.8}}
  }

# Reward function R(s, a)
def reward(s, a):
  return 1.0 if (s == 3 and a == 'h') else 0.0

# Target policy pi^t
pi_t = {
  1: {'g': 0.9, 'h': 0.1},
  2: {'g': 0.9, 'h': 0.1},
  3: {'g': 0.1, 'h': 0.9}
  }

# Behavior policy pi^b
pi_b = {
  1: {'g': 0.85, 'h': 0.15},
  2: {'g': 0.88, 'h': 0.12},
  3: {'g': 0.1, 'h': 0.9}
  }

def compute_value_function(policy):
  """
  Compute the value function V^pi using the Bellman equation.
  """
  n_states = len(states)

  # Expected transition matrix P_pi and expected reward vector R_pi
  P_pi = np.zeros((n_states, n_states))
  R_pi = np.zeros(n_states)

  for i, s in enumerate(states):
    for a in actions:
      pi_sa = policy[s][a]
      R_pi[i] = R_pi[i] + pi_sa * reward(s, a)

      for j, s_prime in enumerate(states):
        P_pi[i, j] = P_pi[i, j] + pi_sa * P[s][a][s_prime]
        # print(P_pi)

  # Solve (I - γP_pi)V = R_pi
  I = np.eye(n_states)
  A = I - gamma * P_pi
  V = solve(A, R_pi)

  return V

V_target_exact = compute_value_function(pi_t)
V_behavior_exact = compute_value_function(pi_b)

# Fancy display to impress everyone
# Compute exact values using model
display(Markdown("**Exact Values (Model-Based)**"))

display(Markdown("**Target Policy $\\pi^t$**"))
for i, s in enumerate(states):
  display(Markdown(f"$V^{{\\pi^t}}({s}) = {V_target_exact[i]:.4f}$"))
  # print(f"  V^pi_t({s}) = {V_target_exact[i]:.4f}")

display(Markdown("**Target Policy $\\pi^b$**"))
for i, s in enumerate(states):
  display(Markdown(f"$V^{{\\pi^b}}({s}) = {V_behavior_exact[i]:.4f}$"))
  # print(f"  V^pi_b({s}) = {V_behavior_exact[i]:.4f}")

**Exact Values (Model-Based)**

**Target Policy $\pi^t$**

$V^{\pi^t}(1) = 10.3676$

$V^{\pi^t}(2) = 10.9210$

$V^{\pi^t}(3) = 11.7842$

**Target Policy $\pi^b$**

$V^{\pi^b}(1) = 10.2405$

$V^{\pi^b}(2) = 10.8086$

$V^{\pi^b}(3) = 11.6824$

**Answer 1.2:**

In [ ]:
epsilon = 0.1

# Compute R_max from reward(s, a) function earlier
R_max = max(reward(s, a) for s in states for a in actions)

def compute_effective_horizon(gamma, epsilon, R_max):
    """
    Compute effective horizon T such that the truncation error < epsilon
    We use the bound:
        error ≤ R_max * gamma^T / (1 - gamma)
    and solve for T:
        T ≥ log(R_max / (epsilon * (1 - gamma))) / log(1 / gamma)
    """
    T = int(
        np.ceil(np.log(R_max / (epsilon * (1 - gamma))) / np.log(1 / gamma))
    )
    return T
# Make fancy display to impress everyone
T_effective = compute_effective_horizon(gamma, epsilon, R_max)
display(Markdown(f" **Effective Horizon T** = {T_effective}"))

 **Effective Horizon T** = 104

**Answer 1.3:**

In [ ]:
def generate_trajectory(start_state, policy, horizon):
  """
  Generate a trajectory of fixed length following the given policy.
  """
  trajectory = []
  state = start_state

  for t in range(horizon):
    # Sample action from policy
    action = np.random.choice(actions, p=[policy[state]['g'], policy[state]['h']])

    # Get reward
    r = reward(state, action)

    # Sample next state
    next_state = np.random.choice(states, p=[P[state][action][s] for s in states])

    trajectory.append((state, action, r))
    state = next_state

  return trajectory

def off_policy_mc_finite(start_state, behavior_policy, target_policy,
                         num_trajectories, horizon):
    """
    Off-policy Monte Carlo evaluation with finite trajectories.
    Uses weighted importance sampling.
    """
    returns = []
    weights = []

    for i in range(num_trajectories):
        trajectory = generate_trajectory(start_state, behavior_policy, horizon)

        # Compute return G for this trajectory
        G = 0
        for t in range(len(trajectory) - 1, -1, -1):
            _, _, r = trajectory[t]
            G = gamma * G + r

        # Compute importance sampling ratio W
        W = 1.0
        for state_t, action_t, _ in trajectory:
            pi_target = target_policy[state_t][action_t]
            pi_behavior = behavior_policy[state_t][action_t]
            W *= pi_target / pi_behavior

        returns.append(G)
        weights.append(W)

    # Weighted importance sampling estimate
    returns = np.array(returns)
    weights = np.array(weights)

    if np.sum(weights) > 0:
        V_hat = np.sum(weights * returns) / np.sum(weights)
    else:
        V_hat = 0.0

    return V_hat, returns, weights

num_trajectories = 50 # Provided
np.random.seed(1)

V_hat_target, returns, weights = off_policy_mc_finite(
    start_state=1,
    behavior_policy=pi_b,
    target_policy=pi_t,
    num_trajectories=num_trajectories,
    horizon=T_effective
)


# Make fancy display to impress everyone
display(Markdown(
    f"### Evaluating target policy  $\\hat V^{{\\pi^t}}$ using importance sampling"
))
display(Markdown(
    f"Estimated value function: $\\hat V^{{\\pi^t}}(1) = {V_hat_target:.4f}$"
))


### Evaluating target policy  $\hat V^{\pi^t}$ using importance sampling

Estimated value function: $\hat V^{\pi^t}(1) = 10.4068$

**Answer 1.4:**

In [ ]:
# True infinite-horizon value from Part 1.1
V_true = V_target_exact[0]

# Estimated value from Part 1.3
V_est = V_hat_target

# Error
abs_error = abs(V_est - V_true)
# Make fancy display to impress everyone
display(Markdown(
    f"""
**True value:**  $V^{{\\pi^t}}(1) = {V_true:.4f},$
**Estimated value:**  $\\hat V^{{\\pi^t}}(1) = {V_est:.4f},$
**and Absolute error:**  ${abs_error:.4f}$
    """
))



**True value:**  $V^{\pi^t}(1) = 10.3676,$
**Estimated value:**  $\hat V^{\pi^t}(1) = 10.4068,$
**and Absolute error:**  $0.0392$
    

**Answer 4.2**

In [ ]:
# State and action spaces
state_space  = np.arange(-3, 4)
action_space = np.arange(-1, 2)

# Given Q-hat
def qhat0(s, a):
  return 2*s**2 + a**2 - s*a + 0.5

# Random tie-breaking greedy action
def greedy_action_with_ties(Q_state):
  values = np.array(list(Q_state.values()))
  actions = list(Q_state.keys())
  max_val = values.max()
  best_idxs = np.where(values == max_val)[0]
  chosen_idx = np.random.choice(best_idxs)
  return actions[chosen_idx]

# Build uniform pi0
pi0 = {s: {a: 1/3 for a in action_space} for s in state_space}

# Build greedy p̄(s)
pbar = {}
for s in state_space:
  Qs = {a: qhat0(s, a) for a in action_space}
  a_star = greedy_action_with_ties(Qs)
  pbar[s] = {a: (1.0 if a == a_star else 0.0) for a in action_space}

# Form π1 = (1 − α)π0 + α p̄
alpha = 0.25
pi1 = {}

for s in state_space:
  pi1[s] = {}
  for a in action_space:
      pi1[s][a] = (1 - alpha)*pi0[s][a] + alpha*pbar[s][a]

# Pretty display
def show_pi1_state(s):
    dist = pi1[s]
    display(Markdown(
        rf"""
### π₁(s = {s})

| action \(a\) | π₁(a ∣ s={s}) |
|-------------:|--------------:|
| -1           | {dist[-1]:.2f} |
|  0           | {dist[0]:.2f} |
|  1           | {dist[1]:.2f} |
"""
    ))

show_pi1_state(-1)
show_pi1_state(0)
show_pi1_state(2)


### π₁(s = -1)

| action \(a\) | π₁(a ∣ s=-1) |
|-------------:|--------------:|
| -1           | 0.25 |
|  0           | 0.25 |
|  1           | 0.50 |



### π₁(s = 0)

| action \(a\) | π₁(a ∣ s=0) |
|-------------:|--------------:|
| -1           | 0.25 |
|  0           | 0.25 |
|  1           | 0.50 |



### π₁(s = 2)

| action \(a\) | π₁(a ∣ s=2) |
|-------------:|--------------:|
| -1           | 0.50 |
|  0           | 0.25 |
|  1           | 0.25 |


**Answer 5.1:**

In [ ]:
gamma = 0.9
epsilon = 0.1

states = ["b", "c", "d", "e"]
actions = ["x", "y"]

Q = {
    ("b", "x"): -1.5,  ("b", "y"): -2.5,
    ("c", "x"): -0.5,  ("c", "y"): -1.0,
    ("d", "x"):  0.0,  ("d", "y"): -0.25,
    ("e", "x"): -0.5,  ("e", "y"):  0.75,
}
def argmax(d):
  best_key = None
  best_val = -np.inf
  for k, v in d.items():
      if v > best_val:
          best_val = v
          best_key = k
  return best_key

def get_epsilon_greedy_policy(Q, states, actions, epsilon):
  policy = {}

  for s in states:
    q_values = {a: Q[(s, a)] for a in actions}
    max_action = argmax(q_values)

    print(f"\nState {s}:")
    print(f"  Q({s}, x) = {Q[(s,'x')]:.4f}, Q({s}, y) = {Q[(s,'y')]:.4f}")
    print(f"  Best action: {max_action}")

    # ε-greedy assignment
    for a in actions:
      if a == max_action:
          policy[(s, a)] = 1 - epsilon + epsilon / len(actions)
      else:
          policy[(s, a)] = epsilon / len(actions)

    print(f"pi(x|{s}) = {policy[(s,'x')]:.4f}, pi(y|{s}) = {policy[(s,'y')]:.4f}")

  return policy

policy = get_epsilon_greedy_policy(Q, states, actions, epsilon)
formatted_policy = {k: round(v, 2) for k, v in policy.items()}
print("")
# Make fancy display to impress everyone
table = "| $State$ | $Action$ | $\\pi(a|s)$ |\n"
table += "|-------|--------|---------|\n"

for s in states:
  for a in actions:
    p = formatted_policy[(s, a)]
    table += f"| {s} | {a} | {p:.2f} |\n"

display(Markdown(table))


State b:
  Q(b, x) = -1.5000, Q(b, y) = -2.5000
  Best action: x
pi(x|b) = 0.9500, pi(y|b) = 0.0500

State c:
  Q(c, x) = -0.5000, Q(c, y) = -1.0000
  Best action: x
pi(x|c) = 0.9500, pi(y|c) = 0.0500

State d:
  Q(d, x) = 0.0000, Q(d, y) = -0.2500
  Best action: x
pi(x|d) = 0.9500, pi(y|d) = 0.0500

State e:
  Q(e, x) = -0.5000, Q(e, y) = 0.7500
  Best action: y
pi(x|e) = 0.0500, pi(y|e) = 0.9500



| $State$ | $Action$ | $\pi(a|s)$ |
|-------|--------|---------|
| b | x | 0.95 |
| b | y | 0.05 |
| c | x | 0.95 |
| c | y | 0.05 |
| d | x | 0.95 |
| d | y | 0.05 |
| e | x | 0.05 |
| e | y | 0.95 |


**Answer 5.2:**

In [ ]:
# Given sub-trajectory
trajectory = [
    ('c', 'x',  1.0, 'e'),
    ('e', 'x', -2.0, 'b'),
    ('b', 'y', -0.5, 'b')
]

def create_dataset(Q, trajectory, gamma):
  """Creates (input, target) pairs for supervised learning."""
  dataset = []

  for (s_t, a_t, r_t, s_next) in trajectory:

    # Input pair
    data_input = (s_t, a_t)

    # Max over next actions: max_a' Q(s_next, a')
    max_q_next = max(Q[(s_next, a)] for a in actions)

    # TD target
    target = r_t + gamma * max_q_next

    dataset.append({
        'input': data_input,
        'target': target
    })

  return dataset


dataset = create_dataset(Q, trajectory, gamma)

# Make fancy display to impress everyone
display(Markdown(f"""**Dataset**`{dataset}` """))


table = "| State | Action | Target |\n"
table += "|-------|--------|---------|\n"

for d in dataset:
  s, a = d["input"]
  y = d["target"]
  table += f"|{s}|{a}|{y:.4f}|\n"

display(Markdown(table))

**Dataset**`[{'input': ('c', 'x'), 'target': 1.675}, {'input': ('e', 'x'), 'target': -3.35}, {'input': ('b', 'y'), 'target': -1.85}]` 

| State | Action | Target |
|-------|--------|---------|
|c|x|1.6750|
|e|x|-3.3500|
|b|y|-1.8500|


**Answer 5.4:**

In [ ]:
# Feature definitions from the problem
def phi1(s, a):
    # phi1(s,a) = -2·1[s=b] - 1·1[s=c] - 0.5·1[s=d]
    return -2.0 * (s == "b") - 1.0 * (s == "c") - 0.5 * (s == "d")

def phi2(s, a):
    # phi2(s,a) = 1·1[a=x] - 1·1[a=y]
    return 1.0 * (a == "x") - 1.0 * (a == "y")

def phi(s, a):
    return np.array([phi1(s, a), phi2(s, a)], dtype=float)

# Build matrix phi and target vector y from the dataset of Part 2
Phi = []
y_vec = []

for d in dataset:
    s, a = d["input"]
    target = d["target"]
    Phi.append(phi(s, a))
    y_vec.append(target)

Phi = np.vstack(Phi)
y_vec = np.array(y_vec)

# Solve least squares: min_theta ||phi theta - y||^2
theta_hat, residuals, rank, svals = np.linalg.lstsq(Phi, y_vec, rcond=None)
theta1_hat, theta2_hat = theta_hat

# Compute predictions and empirical mean square loss
y_pred = Phi @ theta_hat
ms = (1 / len(y_vec)) * np.sum((y_pred - y_vec) ** 2)
# Compute Sum of Squared loss  (without dividing by N as TA mentioned)
ss = np.sum((y_pred - y_vec)**2)


# Make fancy display to impress everyone
display(Markdown(
    rf"""
### Optimal Parameters

$\theta_1^* = {theta1_hat:.6f},$
$\theta_2^* = {theta2_hat:.6f},$
**Mean Squared (MS):**  $L_{{\text{{MS}}}}(\theta^*) = {ms:.6f}$
**and Sum of Squared (SS):** $L_{{\text{{SS}}}}(\theta^*) = {ss:.6f}$
"""
))


table = rf"""### Predictions vs Targets

| State | Action | Target $y$ | Prediction $Q_\theta(s,a)$ |
|-------|--------|-----------|-----------------------------|
"""

for i, d in enumerate(dataset):
  s, a = d["input"]
  y_true = y_vec[i]
  y_hat = y_pred[i]
  table += f"| {s} | {a} | {y_true:.4f} | {y_hat:.4f} |\n"

display(Markdown(table))


### Optimal Parameters

$\theta_1^* = 0.421429,$
$\theta_2^* = -0.082143,$
**Mean Squared (MS):**  $L_{\text{MS}}(\theta^*) = 5.537202$
**and Sum of Squared (SS):** $L_{\text{SS}}(\theta^*) = 16.611607$


### Predictions vs Targets

| State | Action | Target $y$ | Prediction $Q_\theta(s,a)$ |
|-------|--------|-----------|-----------------------------|
| c | x | 1.6750 | -0.5036 |
| e | x | -3.3500 | -0.0821 |
| b | y | -1.8500 | -0.7607 |
